In [2]:
   # pip install requests beautifulsoup4 huggingface_hub rasterio

In [ ]:
"""
01_vs30.py - Descarga el mapa de Vs30 (tipo de suelo) del USGS
Contiene: mapa mundial de Vs30 + recorte de Lima
Nota: ScienceBase está detrás de Cloudflare y bloquea scripts (403 / JSONDecodeError).
Se usa el espejo oficial de USGS ShakeMap (mosaico global de Vs30, 30 arc-seg).
"""
import csv, time
from datetime import datetime
from pathlib import Path
import requests

DATASET = "vs30"
DESTINO = Path("raw/vs30")
LIMA_BBOX = (-77.25, -12.55, -76.60, -11.70)
URL_VS30 = "https://apps.usgs.gov/shakemap_geodata/vs30/global_vs30.grd"


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total, hecho = int(r.headers.get("Content-Length", 0)), 0
        with open(destino, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
                hecho += len(bloque)
                seg = time.perf_counter() - t0
                print(f"\r  ↓ {destino.name}: {hecho/1e6:,.1f}/{total/1e6:,.1f} MB "
                      f"| {seg:,.0f} s | {hecho/1e6/max(seg,0.01):,.2f} MB/s", end="")
    print(f"\n  ✓ {destino.name} en {time.perf_counter()-t0:,.1f} s")


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


# ---------------- DESCARGA ----------------
DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

descargar(URL_VS30, DESTINO / "global_vs30.grd")

# Recorte de Lima
try:
    import rasterio
    from rasterio.windows import from_bounds
    with rasterio.open(DESTINO / "global_vs30.grd") as src:
        ventana = from_bounds(*LIMA_BBOX, transform=src.transform)
        datos = src.read(1, window=ventana)
        perfil = src.profile.copy()
        perfil.update(driver="GTiff", height=datos.shape[0], width=datos.shape[1],
                      transform=src.window_transform(ventana), crs=src.crs or "EPSG:4326",
                      tiled=False)
        perfil.pop("blockxsize", None); perfil.pop("blockysize", None)
        with rasterio.open(DESTINO / "vs30_lima.tif", "w", **perfil) as dst:
            dst.write(datos, 1)
    print("  ✓ Recorte de Lima creado: vs30_lima.tif")
except ImportError:
    print("  (instala rasterio para el recorte de Lima)")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

In [ ]:
"""
02_xbd.py - Descarga xBD (imágenes antes/después con daño de edificios)
Contiene: imágenes satelitales pre/post desastre + etiqueta de daño por edificio
"""
import csv, time
from datetime import datetime
from pathlib import Path
from huggingface_hub import snapshot_download

DATASET = "xbd"
DESTINO = Path("raw/xbd")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


# ---------------- DESCARGA ----------------
DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

snapshot_download(
    repo_id="hannan022/xview2-xbd",
    repo_type="dataset",
    local_dir=DESTINO,
    max_workers=4,          # 4 descargas en paralelo
)

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")
print("Licencia: CC BY-NC-SA 4.0 (uso académico). Citar: Gupta et al., 2019")

In [ ]:
"""
03_open_buildings.py - Descarga los edificios de Lima (Google Open Buildings v3)
Contiene: polígono de cada edificio, área y confianza
"""
import csv, time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
import requests

DATASET = "open_buildings"
DESTINO = Path("raw/open_buildings")
LIMA_BBOX = (-77.25, -12.55, -76.60, -11.70)


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total, hecho = int(r.headers.get("Content-Length", 0)), 0
        with open(destino, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
                hecho += len(bloque)
                seg = time.perf_counter() - t0
                print(f"\r  ↓ {destino.name}: {hecho/1e6:,.1f}/{total/1e6:,.1f} MB "
                      f"| {seg:,.0f} s | {hecho/1e6/max(seg,0.01):,.2f} MB/s", end="")
    print(f"\n  ✓ {destino.name} en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def punto_en_poligono(x, y, anillo):
    dentro, j = False, len(anillo) - 1
    for i in range(len(anillo)):
        xi, yi, xj, yj = anillo[i][0], anillo[i][1], anillo[j][0], anillo[j][1]
        if (yi > y) != (yj > y) and x < (xj - xi) * (y - yi) / (yj - yi) + xi:
            dentro = not dentro
        j = i
    return dentro


def cubre_lima(geom):
    x0, y0, x1, y1 = LIMA_BBOX
    polys = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
    puntos = [(x0, y0), (x0, y1), (x1, y0), (x1, y1), ((x0 + x1) / 2, (y0 + y1) / 2)]
    for poly in polys:
        anillo = poly[0]
        xs, ys = [p[0] for p in anillo], [p[1] for p in anillo]
        if max(xs) < x0 or min(xs) > x1 or max(ys) < y0 or min(ys) > y1:
            continue
        if any(punto_en_poligono(x, y, anillo) for x, y in puntos):
            return True
        if any(x0 <= x <= x1 and y0 <= y <= y1 for x, y in anillo):
            return True
    return False


# ---------------- DESCARGA ----------------
DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

tiles = requests.get("https://openbuildings-public-dot-gweb-research.uw.r.appspot.com/public/tiles.geojson", timeout=120).json()
seleccion = [t for t in tiles["features"] if cubre_lima(t["geometry"])]
print(f"  Teselas que cubren Lima: {len(seleccion)}")

for t in seleccion:
    p = t["properties"]
    url = p.get("tile_url") or ("https://storage.googleapis.com/open-buildings-data/v3/"
                                f"polygons_s2_level_4_gzip/{p['tile_id']}_buildings.csv.gz")
    descargar(url, DESTINO / Path(urlparse(url).path).name)

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

In [ ]:
"""
04_maxar.py - Descarga pares ANTES/DESPUÉS del sismo de Turquía 2023 (Maxar)
Contiene: catálogo de escenas + pares de imágenes .tif (~30 cm/píxel) de la misma zona
Uso: python 04_maxar.py [cantidad_de_pares]
Cada par comparte el mismo quadkey (misma tesela del terreno):
  antes   = imagen previa al 2023-02-06 con menos nubes (la más reciente)
  despues = imagen posterior al sismo con menos nubes (la más cercana al evento)
"""
import csv, io, sys, time
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import requests

DATASET = "maxar_turquia_pares"
DESTINO = Path("raw/maxar_turquia")
EVENTO = "Kahramanmaras-turkey-earthquake-23"
FECHA_SISMO = "2023-02-06"
BASE = "https://raw.githubusercontent.com/opengeos/maxar-open-data/master/datasets"
MAX_PARES = int(sys.argv[1]) if len(sys.argv) > 1 and sys.argv[1].isdigit() else 5


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total, hecho = int(r.headers.get("Content-Length", 0)), 0
        with open(destino, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
                hecho += len(bloque)
                seg = time.perf_counter() - t0
                print(f"\r  ↓ {destino.name}: {hecho/1e6:,.1f}/{total/1e6:,.1f} MB "
                      f"| {seg:,.0f} s | {hecho/1e6/max(seg,0.01):,.2f} MB/s", end="")
    print(f"\n  ✓ {destino.name} en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def nubes(fila):
    return float(fila["tile:clouds_percent"] or 0)


# ---------------- DESCARGA ----------------
DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

# 1) Catálogo de escenas
descargar(f"{BASE}/{EVENTO}.geojson", DESTINO / f"{EVENTO}.geojson")
tsv = requests.get(f"{BASE}/{EVENTO}.tsv", timeout=120).text
(DESTINO / f"{EVENTO}.tsv").write_text(tsv, encoding="utf-8")

# 2) Agrupar escenas por zona (quadkey) y separar antes / después del sismo
zonas = defaultdict(lambda: {"antes": [], "despues": []})
for fila in csv.DictReader(io.StringIO(tsv), delimiter="\t"):
    momento = "antes" if fila["datetime"][:10] < FECHA_SISMO else "despues"
    zonas[fila["quadkey"]][momento].append(fila)

pares = []
for qk, z in zonas.items():
    if not z["antes"] or not z["despues"]:
        continue
    antes = min(z["antes"], key=lambda f: (nubes(f), -int(f["datetime"][:10].replace("-", ""))))
    despues = min(z["despues"], key=lambda f: (nubes(f), f["datetime"]))
    cobertura = float(antes["tile:data_area"] or 0) + float(despues["tile:data_area"] or 0)
    pares.append((nubes(antes) + nubes(despues), -cobertura, qk, antes, despues))
pares.sort(key=lambda p: p[:2])
print(f"  Zonas con antes y después: {len(pares)} | descargando: {min(MAX_PARES, len(pares))} pares")

# 3) Imágenes: raw/maxar_turquia/pares/<quadkey>/{antes,despues}_<fecha>_<catalog_id>.tif
for _, _, qk, antes, despues in pares[:MAX_PARES]:
    print(f"  Zona {qk}: antes {antes['datetime'][:10]} | después {despues['datetime'][:10]}")
    for momento, fila in (("antes", antes), ("despues", despues)):
        nombre = f"{momento}_{fila['datetime'][:10]}_{fila['catalog_id']}.tif"
        descargar(fila["visual"], DESTINO / "pares" / qk / nombre)

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")


In [ ]:
"""
05_microzonificacion.py - Descarga los estudios de microzonificación sísmica de Lima
Contiene: PDFs por distrito con mapas de suelo, isoperiodos y aceleraciones
"""
import csv, time
from datetime import datetime
from pathlib import Path
import requests

DATASET = "microzonificacion"
DESTINO = Path("raw/microzonificacion")

# distrito: número de documento en SIGRID
DISTRITOS = {
    "cercado_de_lima": 197,
    "comas": 642,
    "la_molina": 565,
    "villa_el_salvador": 414,
}


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


# ---------------- DESCARGA ----------------
DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

for distrito, doc_id in DISTRITOS.items():
    try:
        descargar(f"https://sigrid.cenepred.gob.pe/sigridv3/documento/{doc_id}/descargar",
                  DESTINO / f"microzonificacion_{distrito}.pdf")
    except requests.RequestException as e:
        print(f"  ! {distrito}: {e}")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

In [ ]:
"""
06_cismid.py - Descarga acelerogramas de Lima (CISMID - REDACIS)
Contiene: registros de aceleración del suelo (.txt) por estación y por sismo
Cada página de sismo (sismo.php) tiene enlaces directos de descarga por estación;
se guardan solo las estaciones dentro de LIMA_BBOX (según la cabecera del archivo).
"""
import csv, re, time
from datetime import datetime
from pathlib import Path
from urllib.parse import urljoin
import requests

DATASET = "cismid"
DESTINO = Path("raw/cismid")
BASE = "https://www.cismid.uni.edu.pe/ceois/redacis/red/"
LIMA_BBOX = (-77.25, -12.55, -76.60, -11.70)
HEADERS = {"User-Agent": "Mozilla/5.0"}

# carpeta: parámetros de sismo.php (sacados de getevents.php?type=ListaTabla)
SISMOS = {
    "2007-08-15_pisco_Mw7.9":       "event=&codef=2007&evenf=1&gid=512",
    "2025-06-15_callao_M6.1":       "event=225ewhCWcB&codef=&evenf=&gid=3226",
    "2026-05-19_ica_M6.1":          "event=22pxfhs5Ub&codef=&evenf=&gid=3812",
    "2022-01-07_lima_NE_M5.6":      "event=sgxYMOlDXm&codef=&evenf=&gid=1415",
    "2021-11-28_callao_O_M5.2":     "event=mKRyTvgpPv&codef=&evenf=&gid=1402",
}


def coordenadas_estacion(texto):
    # formato 2021-2022: "COORDINATES: lat,lon"; formatos 2007 y 2025+: el primer
    # LATITUDE/LONGITUDE es el de la estación (el segundo es el del epicentro)
    c = re.search(r"COORDINATES\s*:\s*(-?[\d.]+)\s*,\s*(-?[\d.]+)", texto)
    if c:
        return float(c.group(1)), float(c.group(2))
    lat = re.search(r"LATITUDE\s*:\s*(-?[\d.]+)", texto)
    lon = re.search(r"LONGITUDE\s*:\s*(-?[\d.]+)", texto)
    return (float(lat.group(1)), float(lon.group(1))) if lat and lon else None


def en_lima(texto):
    coords = coordenadas_estacion(texto)
    if coords is None:
        return True   # sin coordenadas en la cabecera: se conserva
    lat, lon = coords
    x0, y0, x1, y1 = LIMA_BBOX
    return x0 <= lon <= x1 and y0 <= lat <= y1


def descargar(url, carpeta):
    t0 = time.perf_counter()
    r = requests.get(url, timeout=120, headers=HEADERS)
    r.raise_for_status()
    nombre = re.search(r'filename="?([^";]+)', r.headers.get("Content-Disposition", ""))
    nombre = nombre.group(1) if nombre else url.rsplit("=", 1)[-1] + ".txt"
    destino = carpeta / nombre
    if not en_lima(r.content[:4000].decode("utf-8", "replace")):
        print(f"  - fuera de Lima: {nombre}")
        return
    destino.write_bytes(r.content)
    print(f"  ✓ {nombre} ({len(r.content)/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


# ---------------- DESCARGA ----------------
DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

for carpeta, params in SISMOS.items():
    print(f"\n  Sismo {carpeta}")
    html = requests.get(f"{BASE}sismo.php?{params}", timeout=60, headers=HEADERS).text
    enlaces = list(dict.fromkeys(re.findall(r"href='([^']*download\.php\?[^']+)'", html)))
    enlaces = [e for e in enlaces if "CEOIS" not in e]   # el .zip CEOIS trae todas las estaciones juntas
    print(f"  Estaciones con registro: {len(enlaces)}")
    (DESTINO / carpeta).mkdir(parents=True, exist_ok=True)
    for href in enlaces:
        try:
            descargar(urljoin(BASE, href), DESTINO / carpeta)
        except requests.RequestException as e:
            print(f"  ! {href}: {e}")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")


In [5]:
"""
07_stead.py - Descarga STEAD (ondas sísmicas etiquetadas, Stanford)
Contiene: ondas de 3 componentes + llegada de onda P/S + magnitud
Uso: python 07_stead.py [chunk2 chunk3 ...]   (chunk1 = ruido, chunk2-6 = sismos)
Los enlaces rebrand.ly apuntan a Google Drive, por eso se usa gdown (pip install gdown).
"""
import csv, sys, time
from datetime import datetime
from pathlib import Path
import gdown

DATASET = "stead"
DESTINO = Path("raw/stead")
PARTES = [a for a in sys.argv[1:] if a.startswith("chunk")] or ["chunk2"]
# id de Google Drive de cada parte (destino de https://rebrand.ly/chunkN)
DRIVE_IDS = {
    "chunk1": "1sgLd3eVIrTMcuWtOQRN1fGmiNQ7vc227",
    "chunk2": "1ONNNtiOwFodAAeV-ZWp_-gtWhRcYgU_E",
    "chunk3": "15bt3WS-mQJmkVY4VOb5LYk45xQBYtkoM",
    "chunk4": "12Rw-WI9T21g0hrqtF1KiETYzV4rz177k",
    "chunk5": "1rR6bbqu1yE8N_EVIb3kH3wzPPVLFerFX",
    "chunk6": "1QfONZf8TBuQlkvKjwJgVMgZub4RDUkH9",
}


def descargar(drive_id, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    gdown.download(id=drive_id, output=str(destino), quiet=False)
    print(f"  ✓ {destino.name} en {(time.perf_counter()-t0)/60:,.1f} min")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


# ---------------- DESCARGA ----------------
DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S} | partes: {PARTES}")

for parte in PARTES:
    descargar(DRIVE_IDS[parte], DESTINO / f"{parte}.zip")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")

⏱  Inicio: 23:04:40 | partes: ['-f', 'C:\\Users\\andre\\AppData\\Roaming\\jupyter\\runtime\\kernel-d0906ae1-a2bd-48c1-b9b2-28292c7ad379.json']
  ↓ -f.zip: 0.00/0.00 GB | 0.0 min | 0.01 MB/s
  ✓ -f.zip en 0.0 min
  ↓ kernel-d0906ae1-a2bd-48c1-b9b2-28292c7ad379.json.zip: 0.00/0.00 GB | 0.0 min | 0.01 MB/s
  ✓ kernel-d0906ae1-a2bd-48c1-b9b2-28292c7ad379.json.zip en 0.0 min
⏱  TOTAL stead: 0.05 min | 0.0 MB | 0.00 MB/s


In [ ]:
"""
08_smdtr.py - Cronómetro para la descarga MANUAL de SMD-TR (Turquía)
Contiene: ondas de Turquía + flatfile con PGA, PGV, Sa y Vs30 por estación
DesignSafe exige iniciar sesión, por eso se descarga desde el navegador.
"""
import csv, time
from datetime import datetime
from pathlib import Path

DATASET = "smdtr"
DESTINO = Path("raw/smdtr")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([f"{DATASET} (manual)", datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


# ---------------- CRONÓMETRO ----------------
DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)

print(f"""
PASOS:
  1. Crea tu cuenta gratuita en https://www.designsafe-ci.org (si no la tienes)
  2. Abre https://doi.org/10.17603/ds2-f21x-s189
  3. Descarga el flatfile (CSV) y las carpetas de ondas
  4. Guárdalos en: {DESTINO.resolve()}
""")
input("Presiona ENTER justo cuando EMPIECES a descargar... ")
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")
input("Presiona ENTER cuando TERMINE la descarga... ")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")


PASOS:
  1. Crea tu cuenta gratuita en https://www.designsafe-ci.org (si no la tienes)
  2. Abre https://doi.org/10.17603/ds2-f21x-s189
  3. Descarga el flatfile (CSV) y las carpetas de ondas
  4. Guárdalos en: C:\Datos\BigData\raw\smdtr



In [ ]:
"""
09_shakemap.py - Descarga el ShakeMap del sismo de Turquía 2023 (USGS, M7.8, us6000jllz)
Contiene: grid.xml con PGA, PGV, MMI, Sa(0.3), Sa(1.0), Sa(3.0) y Vs30 en una grilla
La URL de grid.xml se obtiene de la API de eventos de USGS (cambia con cada versión del ShakeMap).
"""
import csv, time
from datetime import datetime
from pathlib import Path
import requests

DATASET = "shakemap_turquia"
DESTINO = Path("raw/shakemap_turquia")
EVENTO = "us6000jllz"
HEADERS = {"User-Agent": "Mozilla/5.0"}


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with requests.get(url, stream=True, timeout=60, headers=HEADERS) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


# ---------------- DESCARGA ----------------
DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

evento = requests.get("https://earthquake.usgs.gov/fdsnws/event/1/query",
                      params={"eventid": EVENTO, "format": "geojson"}, timeout=60, headers=HEADERS).json()
shakemap = evento["properties"]["products"]["shakemap"][0]
print(f"  {evento['properties']['title']} | ShakeMap versión {shakemap['properties'].get('version')}")
descargar(shakemap["contents"]["download/grid.xml"]["url"], DESTINO / "grid.xml")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")


In [ ]:
"""
10_limites_inei.py - Descarga los límites distritales del Perú (INEI)
Contiene: peru_distrital_simple.geojson con los distritos de todo el Perú
(en silver se filtra la provincia de Lima y el Callao)
Fuente: https://github.com/juaneladio/peru-geojson
"""
import csv, time
from datetime import datetime
from pathlib import Path
import requests

DATASET = "limites_inei"
DESTINO = Path("raw/limites_inei")
URL = "https://raw.githubusercontent.com/juaneladio/peru-geojson/master/peru_distrital_simple.geojson"


def descargar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(destino, "wb") as f:
            for bloque in r.iter_content(1024 * 1024):
                f.write(bloque)
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


# ---------------- DESCARGA ----------------
DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

descargar(URL, DESTINO / "peru_distrital_simple.geojson")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")


In [ ]:
"""
11_open_buildings_25d.py - Descarga la ALTURA de edificios de Lima (Google Open Buildings 2.5D Temporal, 2023)
Contiene: GeoTIFFs de altura de edificios (banda building_height) a 4 m, en UTM 18S (EPSG:32718)
Cada tesela original pesa ~390 MB a 0.5 m con 3 bandas; como la resolución efectiva del producto
es 4 m, se lee remotamente SOLO la banda de altura a 4 m (overviews internos del COG) -> ~10 MB por tesela.
Se descargan solo las teselas que tocan los distritos de Lima y Callao (límites INEI, celda 10).
"""
import csv, time
from datetime import datetime
from pathlib import Path
import geopandas as gpd
import rasterio
import requests
from rasterio.enums import Resampling
from shapely.geometry import box

DATASET = "open_buildings_25d"
DESTINO = Path("raw/open_buildings_25d")
BUCKET = "https://storage.googleapis.com/open-buildings-temporal-data"
ANIO = "2023_06_30"
TOKENS_S2 = ["91", "97", "bd"]        # celdas S2 en UTM 18S; Lima cae en "91"
RES_M = 4.0                            # resolución efectiva del producto
LIMITES = Path("raw/limites_inei/peru_distrital_simple.geojson")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([DATASET, datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 1), round(mb / max(seg, 0.01), 2)])


def descargar_altura(url, destino):
    if destino.exists():
        print(f"  ✓ ya existe: {destino.name}")
        return
    t0 = time.perf_counter()
    with rasterio.open(f"/vsicurl/{url}") as src:
        factor = int(RES_M / src.res[0])
        alto, ancho = src.height // factor, src.width // factor
        altura = src.read(2, out_shape=(alto, ancho), resampling=Resampling.nearest)   # banda 2 = building_height
        perfil = dict(driver="GTiff", height=alto, width=ancho, count=1, dtype="float32",
                      crs=src.crs, transform=src.transform * src.transform.scale(factor, factor),
                      nodata=src.nodata, compress="deflate", predictor=3, tiled=True,
                      blockxsize=512, blockysize=512)
    destino.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(destino, "w", **perfil) as dst:
        dst.write(altura, 1)
        dst.set_band_description(1, "building_height_m")
    print(f"  ✓ {destino.name} ({destino.stat().st_size/1e6:,.1f} MB) en {time.perf_counter()-t0:,.1f} s")


# ---------------- DESCARGA ----------------
DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

distritos = gpd.read_file(LIMITES)
lima_utm = distritos[distritos.NOMBPROV.isin(["LIMA", "CALLAO"])].to_crs(32718).union_all()

teselas = []
for token in TOKENS_S2:
    m = requests.get(f"{BUCKET}/v1/manifests/{token}_EPSG_32718_{ANIO}.json", timeout=60).json()
    prefijo = m["uriPrefix"].replace("gs://open-buildings-temporal-data", BUCKET)
    for ts in m["tilesets"]:
        for s in ts["sources"]:
            a, d = s["affineTransform"], s["dimensions"]
            caja = box(a["translateX"], a["translateY"] + a["scaleY"] * d["height"],
                       a["translateX"] + a["scaleX"] * d["width"], a["translateY"])
            if caja.intersects(lima_utm):
                teselas.append(prefijo + s["uris"][0])      # el prefijo se concatena sin "/"
print(f"  Teselas que tocan Lima y Callao: {len(teselas)}")

for url in teselas:
    carpeta, archivo = url.split("/")[-2:]
    descargar_altura(url, DESTINO / f"{carpeta}_{archivo}")

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg/60:,.2f} min | {mb:,.1f} MB | {mb/max(seg,0.01):,.2f} MB/s")
print("Licencia: CC BY 4.0. Citar: Sirko et al., 2023 (Open Buildings 2.5D Temporal)")


In [ ]:
"""
12_norma_e030.py - Parámetros sísmicos de la Norma E.030 Diseño Sismorresistente (RNE, Perú)
Contiene: factor de zona Z, factor de suelo S, periodos TP y TL, y clasificación del suelo por Vs30
No es una descarga: son las tablas de la norma (DS N° 003-2016-VIVIENDA, actualizada por
RM N° 355-2018-VIVIENDA) transcritas a CSV para que entren al lakehouse como una fuente más.
Lima y Callao están en la Zona 4 (Z = 0.45 g).
"""
import csv, time
from datetime import datetime
from pathlib import Path

DATASET = "norma_e030"
DESTINO = Path("raw/norma_e030")

# Tabla N° 1 - Factores de zona
ZONAS = [(4, 0.45), (3, 0.35), (2, 0.25), (1, 0.10)]

# Tabla N° 3 - Factor de suelo S (zona x perfil)
FACTOR_S = {
    4: {"S0": 0.80, "S1": 1.00, "S2": 1.05, "S3": 1.10},
    3: {"S0": 0.80, "S1": 1.00, "S2": 1.15, "S3": 1.20},
    2: {"S0": 0.80, "S1": 1.00, "S2": 1.20, "S3": 1.40},
    1: {"S0": 0.80, "S1": 1.00, "S2": 1.60, "S3": 2.00},
}

# Tabla N° 4 - Periodos TP y TL (s); Tabla N° 2 - Clasificación por Vs30 (m/s)
PERFILES = [
    # perfil, descripcion,              vs30_min, vs30_max, TP,  TL
    ("S0", "Roca dura",                  1500.0,   None,     0.3, 3.0),
    ("S1", "Roca o suelos muy rígidos",   500.0,  1500.0,    0.4, 2.5),
    ("S2", "Suelos intermedios",          180.0,   500.0,    0.6, 2.0),
    ("S3", "Suelos blandos",              None,    180.0,    1.0, 1.6),
]


def escribir(nombre, cabecera, filas):
    with open(DESTINO / nombre, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(cabecera)
        w.writerows(filas)
    print(f"  ✓ {nombre} ({len(filas)} filas)")


def tamano_mb(carpeta):
    return sum(f.stat().st_size for f in Path(carpeta).rglob("*") if f.is_file()) / 1e6


def registrar_tiempo(seg, mb):
    log = Path("raw/tiempos_descarga.csv")
    nuevo = not log.exists()
    with open(log, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nuevo:
            w.writerow(["dataset", "fecha", "segundos", "minutos", "mb", "mb_por_segundo"])
        w.writerow([f"{DATASET} (transcrito)", datetime.now().isoformat(timespec="seconds"),
                    round(seg, 1), round(seg / 60, 2), round(mb, 4), round(mb / max(seg, 0.01), 4)])


# ---------------- GENERACIÓN ----------------
DESTINO.mkdir(parents=True, exist_ok=True)
mb_ini = tamano_mb(DESTINO)
inicio = time.perf_counter()
print(f"⏱  Inicio: {datetime.now():%H:%M:%S}")

escribir("e030_zonas.csv", ["zona", "Z_g"], ZONAS)
escribir("e030_factor_suelo.csv", ["zona", "perfil", "S"],
         [(z, p, s) for z, fila in FACTOR_S.items() for p, s in fila.items()])
escribir("e030_perfiles.csv", ["perfil", "descripcion", "vs30_min", "vs30_max", "TP_s", "TL_s"], PERFILES)

seg = time.perf_counter() - inicio
mb = tamano_mb(DESTINO) - mb_ini
registrar_tiempo(seg, mb)
print(f"⏱  TOTAL {DATASET}: {seg:,.2f} s | {mb*1e3:,.1f} KB")
